In [ ]:
pip install "gymnasium[classic-control]" stable-baselines3 sb3-contrib pyyaml pydantic

In [ ]:
pip uninstall -y torch torchvision torchaudio torchdata torchtext dgl

In [ ]:
pip install torch==2.1.2 

In [ ]:
pip install torchdata==0.7.1

In [ ]:
pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html

In [ ]:
pip install "numpy<2.0.0"

In [ ]:
pip install holidays

In [1]:
from typing import Optional
import datetime
import json
import numpy as np
import pandas as pd
import gymnasium as gym
from pathlib import Path


def _build_special_days(year, n_days):
    import holidays as hl
    pt_holidays = hl.country_holidays("PT", years=[year])
    start = datetime.date(year, 1, 1)
    special = set()
    for d in range(n_days):
        date = start + datetime.timedelta(days=d)
        if date.weekday() == 6 or date in pt_holidays:
            special.add(d)
    return special


class ScheduleEnv(gym.Env):
    def __init__(self, data_dir: str = "../../../../data/problems/SMARTASK_4TEAMS_24EMP"):
        super().__init__()
        base = Path(data_dir)

        with open(base / "problem.json") as f:
            prob = json.load(f)

        self.num_days = prob["temporalScope"]["numDays"]
        self.year = prob["temporalScope"]["year"]
        employees = prob["employees"]["simple"]
        self.num_employees = len(employees)
        self.employee_teams = [set(emp.get("teams", [])) for emp in employees]
        self.dual_team = [len(teams) > 1 for teams in self.employee_teams]

        shifts_sorted = sorted(prob["demand"]["shifts"], key=lambda s: s["order"])
        self.shift_codes = [s["code"] for s in shifts_sorted]
        self.shift_idx = {c: i for i, c in enumerate(self.shift_codes)}
        self.shift_order_map = {s["code"]: s["order"] for s in shifts_sorted}
        self.teams = list(prob["demand"]["organizationalUnits"]["teams"])
        self.team_idx = {t: i for i, t in enumerate(self.teams)}
        self.num_shifts = len(self.shift_codes)
        self.num_teams = len(self.teams)
        self.NUM_ACTIONS = 1 + self.num_shifts * self.num_teams

        self.action_to_shift_team = {}
        self.action_shift_order = {0: 0}
        for t_idx, team in enumerate(self.teams):
            for s_idx, shift in enumerate(self.shift_codes):
                a = 1 + t_idx * self.num_shifts + s_idx
                self.action_to_shift_team[a] = (shift, team)
                self.action_shift_order[a] = self.shift_order_map[shift]

        self.team_sizes = {
            team: sum(1 for s in self.employee_teams if team in s)
            for team in self.teams
        }

        vac_df = pd.read_csv(base / "vacations.csv", header=None)
        self.vac_mask = vac_df.iloc[:, 1:].values.astype(bool)

        dem_df = pd.read_csv(base / "demand.csv")
        dem_df["date"] = pd.to_datetime(dem_df["date"])
        start_ts = pd.Timestamp(f"{self.year}-01-01")
        dem_df["day_idx"] = (dem_df["date"] - start_ts).dt.days

        self.min_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        for _, row in dem_df.iterrows():
            d = int(row["day_idx"])
            s = self.shift_idx[row["shift"]]
            t = self.team_idx[row["team"]]
            self.min_demand[d, s, t] = int(row["minimum"])

        self.special_days = _build_special_days(self.year, self.num_days)

        self.max_days_per_year = 223
        self.max_consecutive_days = 5
        self.special_days_cap = 22

        self.total_steps = self.num_employees * self.num_days
        self.num_features = 2

        self.observation_space = gym.spaces.Box(
            low=-1.0, high=10.0,
            shape=(self.total_steps, self.num_features + self.NUM_ACTIONS),
            dtype=np.float32,
        )
        self.action_space = gym.spaces.Discrete(self.NUM_ACTIONS)
        self.reset()

    def _build_initial_matrix(self):
        matrix = np.zeros(
            (self.total_steps, self.num_features + self.NUM_ACTIONS), dtype=np.float32
        )
        self.emp_day_to_row = {}
        row = 0
        for day in self.day_order:
            for emp in range(self.num_employees):
                matrix[row, 0] = emp
                matrix[row, 1] = day
                self.emp_day_to_row[(emp, day)] = row
                row += 1
        return matrix

    def _row(self, emp, day):
        return self.emp_day_to_row[(emp, day)]

    def _is_working(self, emp, day):
        row = self.emp_day_to_row[(emp, day)]
        slot = self.state[row, 2:]
        return slot.sum() > 0 and slot[0] != 1

    def _get_assigned_shift(self, emp, day):
        row = self.emp_day_to_row[(emp, day)]
        slot = self.state[row, 2:]
        if slot.sum() == 0 or slot[0] == 1:
            return None
        for a in range(1, self.NUM_ACTIONS):
            if slot[a] == 1:
                return self.action_to_shift_team[a][0]
        return None

    def _get_prev_shift(self, emp, day):
        if day == 0:
            return None
        return self._get_assigned_shift(emp, day - 1)

    def _get_next_shift(self, emp, day):
        if day >= self.num_days - 1:
            return None
        return self._get_assigned_shift(emp, day + 1)

    def _consecutive_streak_if_work(self, emp, day):
        streak = 1
        d = day - 1
        while d >= 0 and self._is_working(emp, d):
            streak += 1
            d -= 1
        d = day + 1
        while d < self.num_days and self._is_working(emp, d):
            streak += 1
            d += 1
        return streak

    def _get_info(self):
        return {}

    def current_emp_day(self):
        if self.phase == 1:
            return int(self.state[self.current_step, 0]), int(self.state[self.current_step, 1])
        else:
            return self.phase2_steps[self.phase2_idx]

    def _team_balance_bonus(self, emp_id, team):
        if not self.dual_team[emp_id]:
            return 0.0
        sizes = {t: self.team_sizes[t] for t in self.employee_teams[emp_id]}
        min_size, max_size = min(sizes.values()), max(sizes.values())
        if min_size == max_size:
            return 0.0
        smaller = {t for t, s in sizes.items() if s == min_size}
        return (max_size - min_size) * 0.5 if team in smaller else 0.0

    def _calculate_reward_phase1(self, emp_id, day_id, action):
        reward = 0.0

        if action != 0:
            _, team = self.action_to_shift_team[action]
            reward += self._team_balance_bonus(emp_id, team)

        is_last_emp_today = (
            self.current_step >= self.total_steps or
            int(self.state[self.current_step, 1]) != day_id
        )
        if is_last_emp_today:
            day_shortfall = np.maximum(0, self.min_demand[day_id] - self.daily_coverage[day_id]).sum()
            if day_shortfall == 0:
                reward += 3.0
            else:
                reward -= day_shortfall * 2.0

        return reward

    def _calculate_reward_phase2(self, emp_id, day_id, action):
        reward = 0.0
        day_has_shortfall = np.any(
            self.daily_coverage[day_id] < self.min_demand[day_id]
        )

        if action != 0:
            shift, team = self.action_to_shift_team[action]
            s, t = self.shift_idx[shift], self.team_idx[team]
            cov = self.daily_coverage[day_id, s, t]
            mn = self.min_demand[day_id, s, t]

            if cov <= mn:
                reward += 3.0  # Filling a shortfall gap
            else:
                reward += 1.0  # Non-shortfall day, working toward 223

            reward += self._team_balance_bonus(emp_id, team)
        else:
            if day_has_shortfall:
                reward -= 0.5  # Resting on a shortfall day
            else:
                reward -= 0.1  # Resting on non-shortfall day

        return reward

    def _calculate_final_reward(self):
        reward = 0.0
        total_shortfall = int(np.maximum(0, self.min_demand - self.daily_coverage).sum())
        reward -= total_shortfall * 20.0

        if total_shortfall == 0:
            reward += 500.0

        # Penalize distance from 223 target (secondary goal)
        for emp in range(self.num_employees):
            deficit = self.max_days_per_year - self.days_worked[emp]
            if deficit > 0:
                reward -= deficit * 1.0

        return reward

    def _build_phase2_steps(self):
        shortfall_steps = []
        fill_steps = []

        for day in range(self.num_days):
            has_gap = bool(np.any(self.daily_coverage[day] < self.min_demand[day]))

            for emp in range(self.num_employees):
                if self.days_worked[emp] >= self.max_days_per_year:
                    continue
                row = self._row(emp, day)
                slot = self.state[row, 2:]
                if slot[0] != 1:  # not resting
                    continue

                (shortfall_steps if has_gap else fill_steps).append((emp, day))

        return shortfall_steps + fill_steps

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.day_order = list(range(self.num_days))
        self.state = self._build_initial_matrix()
        self.current_step = 0
        self.phase = 1
        self.phase2_steps = []
        self.phase2_idx = 0
        self.phase1_shortfall = 0
        self.days_worked = np.zeros(self.num_employees, dtype=int)
        self.special_days_worked = np.zeros(self.num_employees, dtype=int)
        self.daily_coverage = np.zeros(
            (self.num_days, self.num_shifts, self.num_teams), dtype=np.float32
        )
        return self.state, self._get_info()

    def step(self, action):
        emp_id, day_id = self.current_emp_day()
        row = self._row(emp_id, day_id)

        if self.phase == 1:
            self.state[row, 2 + action] = 1

            if action != 0:
                self.days_worked[emp_id] += 1
                shift, team = self.action_to_shift_team[action]
                s, t = self.shift_idx[shift], self.team_idx[team]
                self.daily_coverage[day_id, s, t] += 1
                if day_id in self.special_days:
                    self.special_days_worked[emp_id] += 1

            self.current_step += 1
            reward = self._calculate_reward_phase1(emp_id, day_id, action)

            if self.current_step >= self.total_steps:
                self.phase1_shortfall = int(
                    np.maximum(0, self.min_demand - self.daily_coverage).sum()
                )
                self.phase = 2
                self.phase2_steps = self._build_phase2_steps()
                self.phase2_idx = 0

                if len(self.phase2_steps) == 0:
                    reward += self._calculate_final_reward()
                    return self.state.copy(), reward, True, False, self._get_info()

                return self.state.copy(), reward, False, False, self._get_info()

            return self.state.copy(), reward, False, False, self._get_info()

        else:
            if action != 0:
                self.state[row, 2] = 0  # clear REST
                self.state[row, 2 + action] = 1

                self.days_worked[emp_id] += 1
                shift, team = self.action_to_shift_team[action]
                s, t = self.shift_idx[shift], self.team_idx[team]
                self.daily_coverage[day_id, s, t] += 1
                if day_id in self.special_days:
                    self.special_days_worked[emp_id] += 1

            self.phase2_idx += 1
            reward = self._calculate_reward_phase2(emp_id, day_id, action)

            terminated = self.phase2_idx >= len(self.phase2_steps)
            if terminated:
                reward += self._calculate_final_reward()

            return self.state.copy(), reward, terminated, False, self._get_info()

    def _rest_only_mask(self):
        m = np.zeros(self.NUM_ACTIONS, dtype=bool)
        m[0] = True
        return m

    def get_action_mask(self):
        if self.phase == 1 and self.current_step >= self.total_steps:
            return self._rest_only_mask()
        if self.phase == 2 and self.phase2_idx >= len(self.phase2_steps):
            return self._rest_only_mask()

        emp, day = self.current_emp_day()
        mask = np.ones(self.NUM_ACTIONS, dtype=bool)

        if self.vac_mask[emp, day]:
            mask[1:] = False
            return mask

        if self.days_worked[emp] >= self.max_days_per_year:
            mask[1:] = False
            return mask

        if self._consecutive_streak_if_work(emp, day) > self.max_consecutive_days:
            mask[1:] = False
            return mask

        if day in self.special_days and self.special_days_worked[emp] >= self.special_days_cap:
            mask[1:] = False
            return mask

        # No-earlier-shift rule: if prev day's shift had higher order, today
        # cannot do any action with strictly lower order.
        prev_shift = self._get_prev_shift(emp, day)
        if prev_shift is not None:
            prev_order = self.shift_order_map[prev_shift]
            for a in range(1, self.NUM_ACTIONS):
                if self.action_shift_order[a] < prev_order:
                    mask[a] = False

        # Symmetric forward check: today's shift must not exceed tomorrow's order.
        next_shift = self._get_next_shift(emp, day)
        if next_shift is not None:
            next_order = self.shift_order_map[next_shift]
            for a in range(1, self.NUM_ACTIONS):
                if self.action_shift_order[a] > next_order:
                    mask[a] = False

        # Employees can only work for teams listed in their profile.
        for action, (_, team) in self.action_to_shift_team.items():
            if team not in self.employee_teams[emp]:
                mask[action] = False

        if self.phase == 1:
            for action in range(1, self.NUM_ACTIONS):
                if mask[action]:
                    shift, team = self.action_to_shift_team[action]
                    s, t = self.shift_idx[shift], self.team_idx[team]
                    if self.daily_coverage[day, s, t] >= self.min_demand[day, s, t]:
                        mask[action] = False

        if self.phase == 2:
            day_has_gap = np.any(self.daily_coverage[day] < self.min_demand[day])
            if day_has_gap and np.any(mask[1:]):
                can_fill_gap = False
                for a in range(1, self.NUM_ACTIONS):
                    if mask[a]:
                        shift, team = self.action_to_shift_team[a]
                        s, t = self.shift_idx[shift], self.team_idx[team]
                        if self.daily_coverage[day, s, t] < self.min_demand[day, s, t]:
                            can_fill_gap = True
                            break
                if can_fill_gap:
                    mask[0] = False  # force work only when they can fill a gap
                    for a in range(1, self.NUM_ACTIONS):
                        if mask[a]:
                            shift, team = self.action_to_shift_team[a]
                            s, t = self.shift_idx[shift], self.team_idx[team]
                            if self.daily_coverage[day, s, t] >= self.min_demand[day, s, t]:
                                mask[a] = False

        if not np.any(mask):
            mask[0] = True
        return mask

    def action_label(self, action):
        if action == 0:
            return "-"
        shift, team = self.action_to_shift_team[action]
        return f"{shift}-{team}"

    def render(self):
        for emp in range(self.num_employees):
            schedule = []
            for day in range(self.num_days):
                r = self.state[self._row(emp, day), 2:]
                schedule.append(self.action_label(int(np.argmax(r))) if r.sum() > 0 else "-")
            print(f"Employee {emp + 1:2d}: {schedule}")


In [2]:
import dgl
import dgl.nn as dglnn
import torch
import numpy as np


def emp_feat_dim(env):
    # [days_worked/223, streak/5, in_team_X for each team, emp_position]
    return 2 + env.num_teams + 1


def day_feat_dim(env):
    # [cov_gap for each (shift, team), is_special, day_position]
    return env.num_shifts * env.num_teams + 2


def build_graph(env):
    sources = []
    destinations = []
    for emp in range(env.num_employees):
        for day in range(env.num_days):
            sources.append(emp)
            destinations.append(day)

    graph_data = {
        ("employee", "assigned_to", "day"): (torch.tensor(sources), torch.tensor(destinations)),
        ("day", "staffed_by", "employee"): (torch.tensor(destinations), torch.tensor(sources)),
    }
    g = dgl.heterograph(graph_data)
    update_graph_features(g, env)
    return g


def update_graph_features(g, env):
    n_emp_feats = emp_feat_dim(env)
    n_day_feats = day_feat_dim(env)

    # Employee features: [days_worked/223, streak/5, in_team_{T} for each team, emp_position]
    emp_feats = np.zeros((env.num_employees, n_emp_feats), dtype=np.float32)
    for emp in range(env.num_employees):
        emp_feats[emp, 0] = env.days_worked[emp] / 223.0
        emp_feats[emp, 1] = env._consecutive_streak_if_work(emp, env.current_emp_day()[1]) / 5.0
        # one-hot-style team membership flags (multi-hot for dual-team employees)
        for t_idx, team in enumerate(env.teams):
            emp_feats[emp, 2 + t_idx] = float(team in env.employee_teams[emp])
        emp_feats[emp, 2 + env.num_teams] = emp / env.num_employees

    # Day features: [cov_gap for each (shift, team) pair, is_special, day_position]
    # cov_gap order matches env.action_to_shift_team ordering (team outer, shift inner).
    day_feats = np.zeros((env.num_days, n_day_feats), dtype=np.float32)
    for d in range(env.num_days):
        idx = 0
        for t_idx in range(env.num_teams):
            for s_idx in range(env.num_shifts):
                day_feats[d, idx] = env.min_demand[d, s_idx, t_idx] - env.daily_coverage[d, s_idx, t_idx]
                idx += 1
        day_feats[d, idx] = float(d in env.special_days)
        day_feats[d, idx + 1] = d / env.num_days

    g.nodes["employee"].data["feat"] = torch.tensor(emp_feats, dtype=torch.float32)
    g.nodes["day"].data["feat"] = torch.tensor(day_feats, dtype=torch.float32)


def coverage_gap_vector(env, day_id):
    """Build the live coverage-gap vector for the current step.
    Order MUST match the day_feats ordering above (team outer, shift inner)."""
    gaps = []
    for t_idx in range(env.num_teams):
        for s_idx in range(env.num_shifts):
            gaps.append(env.min_demand[day_id, s_idx, t_idx] - env.daily_coverage[day_id, s_idx, t_idx])
    return gaps


/home/joao/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GNNActorCritic(nn.Module):
    def __init__(self, emp_in_feats, day_in_feats, num_actions, cov_gap_dim,
                 hidden_dim=64, encoded_dim=64):
        super().__init__()
        self.encoded_dim = encoded_dim
        self.cov_gap_dim = cov_gap_dim

        # Projects heterogeneous node features (emp_in_feats vs day_in_feats) to a
        # shared hidden_dim so the same GNN layers can be applied to both types.
        self.emp_proj = nn.Linear(emp_in_feats, hidden_dim)
        self.day_proj = nn.Linear(day_in_feats, hidden_dim)

        # First layer: HeteroGraphConv with one SAGEConv per relation
        # (assigned_to: employee->day, staffed_by: day->employee). 'mean' aggregator
        # normalises across the very different node degrees of the two sides.
        self.conv1 = dglnn.HeteroGraphConv({
            'assigned_to': dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean'),
            'staffed_by': dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean'),
        }, aggregate='sum')

        # Second layer: 2-hop reasoning — lets employees see other employees through
        # shared days, and days see other days through shared employees.
        self.conv2 = dglnn.HeteroGraphConv({
            'assigned_to': dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean'),
            'staffed_by': dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean'),
        }, aggregate='sum')

        # Head input: emp_emb + day_emb + live_cov_gap(S*T) + dyn_feats(2) + phase(1)
        head_in = encoded_dim * 2 + cov_gap_dim + 2 + 1

        # Separate heads (no shared MLP layers beyond the trunk) — keeps policy
        # and value gradients from interfering during PPO updates.
        self.actor_head = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.Tanh(),
            nn.Linear(64, num_actions),
        )
        self.critic_head = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    @classmethod
    def from_env(cls, env, hidden_dim=64, encoded_dim=64):
        """Build a model whose input/output dims match the supplied env."""
        return cls(
            emp_in_feats=emp_feat_dim(env),
            day_in_feats=day_feat_dim(env),
            num_actions=env.NUM_ACTIONS,
            cov_gap_dim=env.num_shifts * env.num_teams,
            hidden_dim=hidden_dim,
            encoded_dim=encoded_dim,
        )

    def gnn_forward(self, g):
        # Initial projection of node features to hidden dimension
        h = {
            "employee": self.emp_proj(g.nodes["employee"].data["feat"]),
            "day": self.day_proj(g.nodes["day"].data["feat"]),
        }
        # ReLU between layers — without it conv1∘conv2 collapses to one linear transform.
        h = self.conv1(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        return h["employee"], h["day"]

    def _compute_heads(self, emp_e, day_e, cov_gaps, dyn_feats, action_masks, phase):
        # Normalise dynamic features to [0, 1] using known maxima (223 work-days, 5-day streak).
        norm_dyn = dyn_feats.clone()
        norm_dyn[..., 0] /= 223.0
        norm_dyn[..., 1] /= 5.0
        norm_cov_gaps = cov_gaps / 10.0  # rough max team size for normalisation
        combined = torch.cat([emp_e, day_e, norm_cov_gaps, norm_dyn, phase], dim=-1)
        values = self.critic_head(combined).squeeze(-1)
        logits = self.actor_head(combined)
        # Mask invalid actions with -inf so softmax assigns them exactly 0 probability.
        masks_bool = torch.as_tensor(action_masks, dtype=torch.bool)
        if not masks_bool.any():
            masks_bool[..., 0] = True
        logits = logits.masked_fill(~masks_bool, float("-inf"))
        probs = F.softmax(logits, dim=-1)
        return probs, values

    def _heads(self, emp_emb, day_emb, emp_ids, day_ids,
              cov_gaps, dyn_feats, action_masks, phase):
        # Gather: pick the embeddings for the current (emp, day) step out of the
        # full per-node outputs from the GNN trunk.
        e = emp_emb[emp_ids]
        d = day_emb[day_ids]
        return self._compute_heads(e, d, cov_gaps, dyn_feats, action_masks, phase)

    def forward(self, g, emp_ids, day_ids, cov_gaps, dyn_feats, action_masks, phase):
        emp_emb, day_emb = self.gnn_forward(g)
        return self._heads(emp_emb, day_emb, emp_ids, day_ids,
                          cov_gaps, dyn_feats, action_masks, phase)


In [4]:
from dataclasses import dataclass, field
from torch.distributions import Categorical


@dataclass
class Trajectory:
    emp_ids: list = field(default_factory=list)
    day_ids: list = field(default_factory=list)
    coverage_gaps: list = field(default_factory=list)
    dyn_emp_feats: list = field(default_factory=list)
    action_masks: list = field(default_factory=list)
    phases: list = field(default_factory=list)
    actions: list = field(default_factory=list)
    log_probs_old: list = field(default_factory=list)
    rewards: list = field(default_factory=list)
    values: list = field(default_factory=list)
    # Per-step graph feature snapshots for PPO (restored before each gradient update)
    emp_feats_snapshots: list = field(default_factory=list)
    day_feats_snapshots: list = field(default_factory=list)

    def to_tensors(self):
        return {
            "emp_ids": torch.tensor(self.emp_ids, dtype=torch.long),
            "day_ids": torch.tensor(self.day_ids, dtype=torch.long),
            "coverage_gaps": torch.stack(self.coverage_gaps),
            "dyn_emp_feats": torch.stack(self.dyn_emp_feats),
            "action_masks": torch.stack(self.action_masks),
            "phases": torch.stack(self.phases),
            "actions": torch.tensor(self.actions, dtype=torch.long),
            "log_probs_old": torch.tensor(self.log_probs_old, dtype=torch.float32),
            "rewards": torch.tensor(self.rewards, dtype=torch.float32),
            "values": torch.tensor(self.values, dtype=torch.float32),
            "emp_feats": torch.stack(self.emp_feats_snapshots),
            "day_feats": torch.stack(self.day_feats_snapshots),
        }


def _get_consecutive_days(env, emp_id, day_id):
    streak = 0
    d = day_id - 1
    while d >= 0 and env._is_working(emp_id, d):
        streak += 1
        d -= 1
    return streak


def collect_trajectory(env, model, graph):
    model.eval()
    traj = Trajectory()
    env.reset()
    terminated = truncated = False

    cached_day_id = None
    cached_emp_emb = None
    cached_day_emb = None
    cached_emp_feats = None
    cached_day_feats = None

    with torch.no_grad():
        while not (terminated or truncated):
            emp_id, day_id = env.current_emp_day()

            # GNN embeddings only depend on per-day state (graph features change daily),
            # so we recompute once per day rather than once per step.
            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb = model.gnn_forward(graph)
                cached_day_id = day_id
                # clone() — graph tensors get overwritten in-place each day.
                cached_emp_feats = graph.nodes["employee"].data["feat"].clone()
                cached_day_feats = graph.nodes["day"].data["feat"].clone()

            # "Live" features that change every step (cov gaps + per-emp dyn feats),
            # fed to the heads directly rather than re-running the GNN trunk.
            cov_gap = coverage_gap_vector(env, day_id)
            consec = _get_consecutive_days(env, emp_id, day_id)
            dyn_feat = [env.days_worked[emp_id], consec]

            action_mask = env.get_action_mask()
            phase_val = float(env.phase - 1)

            emp_id_t = torch.tensor([emp_id], dtype=torch.long)
            day_id_t = torch.tensor([day_id], dtype=torch.long)
            cov_gap_t = torch.tensor([cov_gap], dtype=torch.float32)
            dyn_feat_t = torch.tensor([dyn_feat], dtype=torch.float32)
            mask_t = torch.tensor(action_mask.tolist(), dtype=torch.bool).unsqueeze(0)
            phase_t = torch.tensor([[phase_val]], dtype=torch.float32)

            probs, values = model._heads(
                cached_emp_emb, cached_day_emb,
                emp_id_t, day_id_t,
                cov_gap_t, dyn_feat_t, mask_t, phase_t
            )

            # Stochastic action selection — masking guarantees no invalid action is sampled.
            dist = Categorical(probs=probs[0])
            action = dist.sample()

            _, reward, terminated, truncated, _ = env.step(action.item())

            traj.emp_ids.append(emp_id)
            traj.day_ids.append(day_id)
            traj.coverage_gaps.append(cov_gap_t[0])
            traj.dyn_emp_feats.append(dyn_feat_t[0])
            traj.action_masks.append(mask_t[0])
            traj.phases.append(phase_t[0])
            traj.actions.append(action.item())
            traj.log_probs_old.append(dist.log_prob(action).item())
            traj.rewards.append(float(reward))
            traj.values.append(values[0].item())
            traj.emp_feats_snapshots.append(cached_emp_feats)
            traj.day_feats_snapshots.append(cached_day_feats)

    return traj


In [5]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    rewards_a = np.array(rewards, dtype=np.float32)
    values_a = np.array(values,  dtype=np.float32)
    values_ext = np.append(values_a, 0.0)

    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards_a[t] + gamma * values_ext[t + 1] - values_ext[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae

    returns = advantages + values_a
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-5)

    return (
        torch.tensor(advantages, dtype=torch.float32),
        torch.tensor(returns, dtype=torch.float32),
    )


In [6]:
from torch.utils.data.sampler import BatchSampler, SubsetRandomSampler


def ppo_update(model, optimizer, batch, advantages, returns, graph,
               clip_eps=0.2, value_coeff=0.5, entropy_coeff=0.01,
               K_epochs=4, mini_batch_size=512):
    T = batch["actions"].shape[0]
    losses = []
    track_entropy = []
    model.train()

    for _ in range(K_epochs):
        for index in BatchSampler(SubsetRandomSampler(range(T)), mini_batch_size, False):
            idx = torch.tensor(index)

            # Restore graph features from snapshot and run GNN
            graph.nodes["employee"].data["feat"] = batch["emp_feats"][idx[0]]
            graph.nodes["day"].data["feat"] = batch["day_feats"][idx[0]]
            emp_emb, day_emb = model.gnn_forward(graph)

            # Recompute action probabilities and state values for the sampled batch using the current model parameters and the cached GNN embeddings
            probs_new, values_new = model._heads(
                emp_emb, day_emb,
                batch["emp_ids"][idx],
                batch["day_ids"][idx],
                batch["coverage_gaps"][idx],
                batch["dyn_emp_feats"][idx],
                batch["action_masks"][idx],
                batch["phases"][idx],
            )

            dist_now = Categorical(probs=probs_new)
            a_logprob_now = dist_now.log_prob(batch["actions"][idx])
            dist_entropy = dist_now.entropy()

            ratios = torch.exp(a_logprob_now - batch["log_probs_old"][idx])
            adv = advantages[idx]
            surr1 = ratios * adv
            surr2 = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps) * adv
            actor_loss = -torch.min(surr1, surr2).mean() - entropy_coeff * dist_entropy.mean()

            critic_loss = F.smooth_l1_loss(values_new, returns[idx])

            loss = actor_loss + value_coeff * critic_loss

            track_entropy.append(dist_entropy.mean().item())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            losses.append(loss.item())

    return float(np.mean(losses)), float(np.mean(track_entropy))


In [8]:
import os

# Hyperparameters
NUM_EPISODES = 10000
GAMMA = 0.99
LAM = 0.95
CLIP_EPS = 0.15
VALUE_COEFF = 0.5
ENTROPY_COEFF_0 = 0.05
ENTROPY_MIN = 0.005
K_EPOCHS = 3
MINI_BATCH_SIZE = 512
LR = 3e-4
SAVE_EVERY = 50

BEST_CKPT = "best_4teams_v1.pth"
LATEST_CKPT = "latest_4teams_v1.pth"
RESUME_FROM = LATEST_CKPT  # set to None to start fresh

env = ScheduleEnv()
graph = build_graph(env)
# Model dims are derived from the env so any scenario size works without code changes.
model = GNNActorCritic.from_env(env)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-5)

print(f"Scenario: {env.num_teams} teams ({env.teams}), {env.num_shifts} shifts ({env.shift_codes}), "
      f"{env.num_employees} employees, NUM_ACTIONS={env.NUM_ACTIONS}")

best_reward = -float("inf")
start_episode = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_episode = ckpt["episode"]
    best_reward = float(ckpt.get("best_reward", -float("inf")))
    print(f"Resumed from episode {start_episode}, best reward = {best_reward:.1f}")

for episode in range(start_episode, NUM_EPISODES):
    entropy_coeff = max(ENTROPY_MIN, ENTROPY_COEFF_0 * (0.999 ** episode))

    traj = collect_trajectory(env, model, graph)
    batch = traj.to_tensors()

    advantages, returns = compute_gae(traj.rewards, traj.values, GAMMA, LAM)

    mean_loss, mean_entropy = ppo_update(
        model, optimizer, batch, advantages, returns, graph,
        clip_eps=CLIP_EPS,
        value_coeff=VALUE_COEFF,
        entropy_coeff=entropy_coeff,
        K_epochs=K_EPOCHS,
        mini_batch_size=MINI_BATCH_SIZE,
    )

    total_reward = sum(traj.rewards)
    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    p1_shortfall = env.phase1_shortfall
    p2_steps = len(env.phase2_steps)
    days_worked_str = ",".join(str(int(d)) for d in env.days_worked)
    mean_days = env.days_worked.mean()

    if total_reward > best_reward:
        best_reward = float(total_reward)
        torch.save(
            {"episode": episode, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_reward": best_reward},
            BEST_CKPT,
        )
        print(f"  NEW BEST: {best_reward:.1f} at ep {episode} "
              f"| shortfall={shortfall} (P1={p1_shortfall}) "
              f"| P2_steps={p2_steps} | days={mean_days:.0f}")

    if episode % 10 == 0:
        print(f"Ep {episode:>4} | R={total_reward:>8.1f} | Best={best_reward:>8.1f} "
              f"| Loss={mean_loss:.4f} | ent={mean_entropy:.4f} "
              f"| shortfall={shortfall} (P1={p1_shortfall}) | P2={p2_steps} "
              f"| days={mean_days:.0f} [{days_worked_str}]")

    if (episode + 1) % SAVE_EVERY == 0:
        torch.save(
            {"episode": episode + 1, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_reward": best_reward},
            LATEST_CKPT,
        )


Scenario: 4 teams (['A', 'B', 'C', 'D']), 2 shifts (['M', 'T']), 24 employees, NUM_ACTIONS=9
Resumed from episode 50, best reward = 2642.8


/home/joao/.local/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Ep   50 | R=  2493.8 | Best=  2642.8 | Loss=0.8607 | ent=0.3634 | shortfall=27 (P1=48) | P2=5462 | days=223 [223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223]
Ep   60 | R=  2463.4 | Best=  2642.8 | Loss=0.8219 | ent=0.3792 | shortfall=27 (P1=47) | P2=5461 | days=223 [223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223]
Ep   70 | R=  2408.6 | Best=  2642.8 | Loss=0.8404 | ent=0.3781 | shortfall=29 (P1=52) | P2=5466 | days=223 [223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223]
Ep   80 | R=  2395.0 | Best=  2642.8 | Loss=0.8367 | ent=0.3826 | shortfall=31 (P1=54) | P2=5468 | days=223 [223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223]
Ep   90 | R=  2431.4 | Best=  2642.8 | Loss=0.8039 | ent=0.3688 | shortfall=30 (P1=48) | P2=5462 | days=223 [223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223,223

KeyboardInterrupt: 

In [9]:
def evaluate(model, env, graph, greedy=False):
    model.eval()
    env.reset()
    terminated = truncated = False
    total_reward = 0.0

    cached_day_id = None
    cached_emp_emb = None
    cached_day_emb = None

    with torch.no_grad():
        while not (terminated or truncated):
            emp_id, day_id = env.current_emp_day()

            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb = model.gnn_forward(graph)
                cached_day_id = day_id

            cov_gap = coverage_gap_vector(env, day_id)
            consec = _get_consecutive_days(env, emp_id, day_id)
            dyn_feat = [env.days_worked[emp_id], consec]

            action_mask = env.get_action_mask()
            phase_val = float(env.phase - 1)

            emp_id_t = torch.tensor([emp_id], dtype=torch.long)
            day_id_t = torch.tensor([day_id], dtype=torch.long)
            cov_gap_t = torch.tensor([cov_gap], dtype=torch.float32)
            dyn_feat_t = torch.tensor([dyn_feat], dtype=torch.float32)
            mask_t = torch.tensor(action_mask.tolist(), dtype=torch.bool).unsqueeze(0)
            phase_t = torch.tensor([[phase_val]], dtype=torch.float32)

            probs, _ = model._heads(
                cached_emp_emb, cached_day_emb,
                emp_id_t, day_id_t,
                cov_gap_t, dyn_feat_t, mask_t, phase_t
            )

            action = probs[0].argmax() if greedy else Categorical(probs=probs[0]).sample()
            _, reward, terminated, truncated, _ = env.step(action.item())
            total_reward += reward

    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())

    snapshot = {
        "phase1_shortfall": env.phase1_shortfall,
        "phase2_steps_len": len(env.phase2_steps),
        "daily_coverage": env.daily_coverage.copy(),
    }

    schedule = []
    for emp in range(env.num_employees):
        row = []
        for day in range(env.num_days):
            r = env.state[env._row(emp, day), 2:]
            row.append(env.action_label(int(np.argmax(r))) if r.sum() > 0 else "-")
        schedule.append(row)

    return total_reward, shortfall, env.days_worked.tolist(), schedule, snapshot


ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Checkpoint: episode {ckpt['episode']}, best reward = {ckpt['best_reward']:.1f}\n")

# Generate N schedules
N = 20
results = []
for i in range(N):
    np.random.seed(42 + i)
    reward, shortfall, days_worked, schedule, snap = evaluate(model, env, graph, greedy=False)
    results.append((reward, shortfall, days_worked, schedule, snap))
    print(f"Run {i+1:2d} | Reward: {reward:8.1f} | Shortfall: {shortfall:3d} "
          f"| P1_shortfall: {snap['phase1_shortfall']:3d} "
          f"| P2_steps: {snap['phase2_steps_len']:3d} "
          f"| Days worked: {days_worked} (avg={np.mean(days_worked):.0f})")

# Pick the run with the lowest shortfall
best_idx = min(range(N), key=lambda i: results[i][1])
best_reward, best_shortfall, best_days, best_schedule, best_snap = results[best_idx]

print(f"\n{'='*60}")
print(f"BEST: Run {best_idx+1} | Reward: {best_reward:.1f} | Shortfall: {best_shortfall}")
print(f"P1 shortfall: {best_snap['phase1_shortfall']}")
print(f"Days worked/employee: {best_days} (avg={np.mean(best_days):.0f})")
print(f"{'='*60}\n")

# Per-day shortfall breakdown for the best run, naming every (shift, team) gap.
best_coverage = best_snap["daily_coverage"]
for d in range(env.num_days):
    gaps = []
    for s_idx, shift in enumerate(env.shift_codes):
        for t_idx, team in enumerate(env.teams):
            gap = int(env.min_demand[d, s_idx, t_idx] - best_coverage[d, s_idx, t_idx])
            if gap > 0:
                gaps.append(f"{shift}-{team}={gap}")
    if gaps:
        print(f"Day {d:3d}: {', '.join(gaps)}")

print()
for emp in range(len(best_schedule)):
    print(f"Employee {emp+1:2d}: {best_schedule[emp]}")

import csv
schedule_csv = f"best_schedule_{env.num_teams}teams_v1.csv"
with open(schedule_csv, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["Employee"] + [f"Day_{d}" for d in range(len(best_schedule[0]))]
    writer.writerow(header)
    for emp in range(len(best_schedule)):
        writer.writerow([f"Employee_{emp+1}"] + best_schedule[emp])

print(f"\nBest schedule written to {schedule_csv}")


Checkpoint: episode 49, best reward = 2642.8

Run  1 | Reward:   2324.0 | Shortfall:  32 | P1_shortfall:  61 | P2_steps: 5475 | Days worked: [223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)
Run  2 | Reward:   2351.0 | Shortfall:  32 | P1_shortfall:  56 | P2_steps: 5470 | Days worked: [223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223, 223] (avg=223)


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
_, _, days_worked_list, _, snap = evaluate(model, env, graph, greedy=False)

days   = np.arange(env.num_days)
cov    = snap["daily_coverage"]   # (num_days, num_shifts, num_teams)
demand = env.min_demand           # (num_days, num_shifts, num_teams)

# Coverage-vs-demand grid: one subplot per (shift, team) pair.
fig, axes = plt.subplots(
    env.num_shifts, env.num_teams,
    figsize=(4 * env.num_teams, 4 * env.num_shifts),
    sharex=True, sharey=True, squeeze=False,
)
for s_idx, shift in enumerate(env.shift_codes):
    for t_idx, team in enumerate(env.teams):
        ax = axes[s_idx][t_idx]
        ax.plot(days, cov[:, s_idx, t_idx], label='Coverage', alpha=0.7)
        ax.plot(days, demand[:, s_idx, t_idx], label='Demand', alpha=0.7, linestyle='--', color='r')
        ax.fill_between(
            days, cov[:, s_idx, t_idx], demand[:, s_idx, t_idx],
            where=cov[:, s_idx, t_idx] < demand[:, s_idx, t_idx],
            color='red', alpha=0.2, label='Shortfall',
        )
        ax.set_title(f"{shift}-{team}")
        if t_idx == 0:
            ax.set_ylabel('Employees')
        if s_idx == env.num_shifts - 1:
            ax.set_xlabel('Day of year')
        ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Coverage vs Demand across the year', fontsize=14)
plt.tight_layout()
plt.show()

# Quarterly violation breakdown — one column per (shift, team) pair plus a total.
pair_labels = [f"{s}-{t}" for t in env.teams for s in env.shift_codes]
header = f"{'Quarter':<12} " + " ".join(f"{p:>8}" for p in pair_labels) + f" {'Total':>6}"
print()
print(header)
print("-" * len(header))
total_viol = 0
for q, (start, end) in enumerate([(0, 91), (91, 182), (182, 273), (273, 365)], 1):
    per_pair = []
    q_total = 0
    for team in env.teams:
        for shift in env.shift_codes:
            s_idx, t_idx = env.shift_idx[shift], env.team_idx[team]
            v = int(np.sum(np.maximum(0, demand[start:end, s_idx, t_idx] - cov[start:end, s_idx, t_idx])))
            per_pair.append(v)
            q_total += v
    total_viol += q_total
    print(f"Q{q} ({start:3d}-{end:3d})  " + " ".join(f"{v:>8}" for v in per_pair) + f" {q_total:>6}")
print("-" * len(header))
print(f"{'Total':<12} " + " ".join(f"{'':>8}" for _ in pair_labels) + f" {total_viol:>6}")

days_worked_arr = np.array(days_worked_list)
print(f"\nDays worked/employee: {days_worked_list}")
print(f"Mean: {days_worked_arr.mean():.1f}, Min: {days_worked_arr.min()}, Max: {days_worked_arr.max()}")
